In [ ]:
!nvidia-smi
!pip install -q transformers datasets peft accelerate bitsandbytes trl wandb

In [ ]:
from tqdm import tqdm
from google.colab import userdata
import dotenv
import os
import wandb

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig

In [ ]:
from huggingface_hub import login
login()

In [ ]:
model_name = "mistralai/Mistral-7B-v0.1"

wandb_api_key = userdata.get('WANDB_API_KEY')
os.environ["WANDB_API_KEY"] = wandb_api_key
wandb.login()


In [ ]:
wandb.init(project="price-predictor-llm")

In [ ]:
from datasets import load_dataset

dataset = load_dataset("ed-donner/items_prompts_lite")

print(dataset)

In [ ]:
def format_example(example):
    return {
        "text": example["prompt"] + example["completion"]
    }

dataset = dataset.map(format_example)

Configure LoRA fine-tuning

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="price-model",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=10,
    save_steps=200,
    fp16=True,
    report_to="wandb"
)

In [ ]:

import inspect
# print(inspect.signature(SFTTrainer))

sft_config = SFTConfig(
    output_dir="price-model",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=10,
    save_steps=200,
    report_to="wandb",
     bf16=True,
    completion_only_loss=False,
)

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset["train"],
    processing_class=tokenizer,
    formatting_func=formatting_func
)

In [ ]:
# tokenizer.pad_token = tokenizer.eos_token
trainer.train()

In [ ]:
import math
import torch
import matplotlib.pyplot as plt

# Color mapping for visualization
COLOR_MAP = {"green": "\033[92m", "orange": "\033[93m", "red": "\033[91m"}
RESET = "\033[0m"

# -----------------------------
# Predictor function
# -----------------------------
def model_predictor(text, model=model, tokenizer=tokenizer, device="cuda"):
    # Encode the prompt
    inputs = tokenizer(text, return_tensors="pt").to(device)
    # Generate output tokens
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=20)
    # Decode and extract the numeric price
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Simple numeric extraction (assumes price is a number in the text)
    import re
    match = re.search(r"\d+\.?\d*", decoded.replace(",", ""))
    if match:
        return float(match.group())
    else:
        return 0.0  # fallback if no number found

# -----------------------------
# Evaluation class
# -----------------------------
class Tester:
    def __init__(self, predictor, data, title=None, size=250):
        self.predictor = predictor
        self.data = data
        self.size = min(size, len(data))
        self.title = title or predictor.__name__.replace("_", " ").title()
        self.guesses = []
        self.truths = []
        self.errors = []
        self.sles = []
        self.colors = []

    def color_for(self, error, truth):
        if error < 40 or error / truth < 0.2:
            return "green"
        elif error < 80 or error / truth < 0.4:
            return "orange"
        else:
            return "red"

    def run_datapoint(self, i):
        datapoint = self.data[i]
        guess = self.predictor(datapoint["text"])
        truth = datapoint["price"]
        error = abs(guess - truth)
        log_error = math.log(truth + 1) - math.log(guess + 1)
        sle = log_error ** 2
        color = self.color_for(error, truth)
        title = datapoint["text"].split("\n\n")[1][:20] + "..." if "\n\n" in datapoint["text"] else datapoint["text"][:20]+"..."
        self.guesses.append(guess)
        self.truths.append(truth)
        self.errors.append(error)
        self.sles.append(sle)
        self.colors.append(color)
        print(f"{COLOR_MAP[color]}{i+1}: Guess: ${guess:,.2f} Truth: ${truth:,.2f} Error: ${error:,.2f} SLE: {sle:,.2f} Item: {title}{RESET}")

    def chart(self, title):
        plt.figure(figsize=(12, 8))
        max_val = max(max(self.truths), max(self.guesses))
        plt.plot([0, max_val], [0, max_val], color='deepskyblue', lw=2, alpha=0.6)
        plt.scatter(self.truths, self.guesses, s=15, c=self.colors)
        plt.xlabel('Ground Truth')
        plt.ylabel('Model Estimate')
        plt.xlim(0, max_val)
        plt.ylim(0, max_val)
        plt.title(title)
        plt.show()

    def report(self):
        average_error = sum(self.errors) / self.size
        rmsle = math.sqrt(sum(self.sles) / self.size)
        hits = sum(1 for color in self.colors if color == "green")
        title = f"{self.title} Error=${average_error:,.2f} RMSLE={rmsle:,.2f} Hits={hits / self.size * 100:.1f}%"
        self.chart(title)

    def run(self):
        for i in range(self.size):
            self.run_datapoint(i)
        self.report()

    @classmethod
    def test(cls, predictor, data, size=250):
        cls(predictor, data, size=size).run()

In [ ]:
# Example: evaluate on first 100 items
Tester.test(model_predictor, dataset["test"], size=100)